# KNN (K-Nearest Neighbors) Algorithm with Classification - Churn Dataset

KNN algoritmasının sınıflandırma amacıyla kullanımı asagıdaki adımlarla ozetlenebilir:
1. Bir k degeri (komsu sayısı) belirlenir.
2. Yeni gelen test verisi icin egitim veri setindeki tum orneklerle mesafe hesaplanır.
3. Mesafesi en yakın k ornek secilir.
4. Bu k kom¸sunun sınıf etiketleri sayılarak en sık gorulen sınıf etiketi tahmin edilir.

KNN algoritması asagıdaki durumlarda iyi calısır:
- Veri boyutu cok yuksek degilse
- Ozellikler benzer olcekteyse (ozellik muhendisligi ve olcekleme yapılmıssa)
- Sınıflar arasındaki ayrım netse

> KNN, özellikle sınıflar arasında belirgin bir ayrım olduğunda, küçük veri setlerinde ve düşük boyutlu özelliklerde iyi performans gösterebilir, ancak büyük veri setlerinde ve yüksek boyutlu özelliklerde performansı düşebilir. Bu nedenle, modelin performansını değerlendirmek için uygun metrikler kullanılarak test verisi üzerinde değerlendirme yapılmalıdır.

In [29]:
import pandas as pd # Veri analizi ve manipülasyonu için kullanılan bir kütüphane
import seaborn as sns # Veri görselleştirme için kullanılan bir kütüphane
import matplotlib.pyplot as plt # Grafik çizimi için kullanılan bir kütüphane

from sklearn.preprocessing import LabelBinarizer, OrdinalEncoder, StandardScaler, MinMaxScaler, RobustScaler # Veriyi sayısal hale getirmek ve ölçeklendirmek için kullanılan araçlar
from sklearn.compose import ColumnTransformer # Verilerin farklı türlerine farklı ön işleme adımları uygulamak için kullanılan bir araç
from sklearn.model_selection import train_test_split # Veriyi eğitim ve test setlerine bölmek için kullanılan bir fonksiyon
from sklearn.neighbors import KNeighborsClassifier # KNN algoritmasını uygulamak için kullanılan bir sınıflandırıcı
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, f1_score # Model performansını değerlendirmek için kullanılan metrikler

In [30]:
df = pd.read_pickle(
    filepath_or_buffer='data/churndata.pkl'
) # Veri setini 'data/churndata.pkl' dosyasından yükler ve df adlı bir DataFrame oluşturur. 
# Bu DataFrame, müşteri kaybı (churn) analizi için kullanılacak verileri içerir.

df.head()

,id,months,offer,phone,multiple,internet_type,gb_mon,security,backup,protection,...,unlimited,contract,paperless,payment,monthly,total_revenue,satisfaction,churn_value,churn_score,cltv
0,8779-QRDMV,1,None,No,No,DSL,8,No,No,Yes,...,No,Month-to-Month,Yes,Bank Withdrawal,39.65,59.65,3,1,91,5433
1,7495-OOKFY,8,Offer E,Yes,Yes,Fiber Optic,17,No,Yes,No,...,Yes,Month-to-Month,Yes,Credit Card,80.65,1024.10,3,1,69,5302
2,1658-BYGOY,18,Offer D,Yes,Yes,Fiber Optic,52,No,No,No,...,Yes,Month-to-Month,Yes,Bank Withdrawal,95.45,1910.88,2,1,81,3179
3,4598-XLKNJ,25,Offer C,Yes,No,Fiber Optic,12,No,Yes,Yes,...,Yes,Month-to-Month,Yes,Bank Withdrawal,98.50,2995.07,2,1,88,5337
4,4846-WHAFZ,37,Offer C,Yes,Yes,Fiber Optic,14,No,No,No,...,Yes,Month-to-Month,Yes,Bank Withdrawal,76.50,3102.36,2,1,67,2793


In [31]:
df.shape # Veri setinin satır ve sütun sayısını gösterir. df.shape[0] satır sayısını, df.shape[1] ise sütun sayısını verir.

(7043, 21)

In [32]:
df.info() # Veri setindeki sütunların veri tiplerini ve eksik değerlerin sayısını gösterir.

<class 'pandas.core.frame.DataFrame'>
Int64Index: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             7043 non-null   object 
 1   months         7043 non-null   int64  
 2   offer          7043 non-null   object 
 3   phone          7043 non-null   object 
 4   multiple       7043 non-null   object 
 5   internet_type  7043 non-null   object 
 6   gb_mon         7043 non-null   int64  
 7   security       7043 non-null   object 
 8   backup         7043 non-null   object 
 9   protection     7043 non-null   object 
 10  support        7043 non-null   object 
 11  unlimited      7043 non-null   object 
 12  contract       7043 non-null   object 
 13  paperless      7043 non-null   object 
 14  payment        7043 non-null   object 
 15  monthly        7043 non-null   float64
 16  total_revenue  7043 non-null   float64
 17  satisfaction   7043 non-null   int64  
 18  churn_va

In [33]:
df.isnull().sum() # Veri setindeki her bir sütunda eksik değerlerin sayısını gösterir.

id               0
months           0
offer            0
phone            0
multiple         0
internet_type    0
gb_mon           0
security         0
backup           0
protection       0
support          0
unlimited        0
contract         0
paperless        0
payment          0
monthly          0
total_revenue    0
satisfaction     0
churn_value      0
churn_score      0
cltv             0
dtype: int64

In [34]:
df.drop(
    columns=['id', 'phone', 'total_revenue', 'cltv', 'churn_score'],
    axis=1,
    inplace=True
)

df.shape

(7043, 16)

In [35]:
round(df.describe(), 2) # Veri setindeki sayısal sütunların temel istatistiklerini gösterir. round() fonksiyonu, istatistiklerin daha okunabilir olması için sonuçları 2 ondalık basamağa yuvarlar.

,months,gb_mon,monthly,satisfaction,churn_value
count,7043.00,7043.00,7043.00,7043.00,7043.00
mean,32.39,20.52,64.76,3.24,0.27
std,24.54,20.42,30.09,1.20,0.44
min,1.00,0.00,18.25,1.00,0.00
25%,9.00,3.00,35.50,3.00,0.00
50%,29.00,17.00,70.35,3.00,0.00
75%,55.00,27.00,89.85,4.00,1.00
max,72.00,85.00,118.75,5.00,1.00


In [36]:
df.describe(include='object') # Veri setindeki kategorik sütunların temel istatistiklerini gösterir. include='object' ifadesi, sadece kategorik sütunları seçmek için kullanılır. Bu, her bir kategorik sütunun benzersiz değerlerini, en sık görülen değeri (top), ve bu değerin frekansını (freq) içerir.

,offer,multiple,internet_type,security,backup,protection,support,unlimited,contract,paperless,payment
count,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043
unique,6,2,4,2,2,2,2,2,3,2,3
top,None,No,Fiber Optic,No,No,No,No,Yes,Month-to-Month,Yes,Bank Withdrawal
freq,3877,4072,3035,5024,4614,4621,4999,4745,3610,4171,3909


In [37]:
# Benzersiz değerlerin sayısını göstermek, her bir kategorik sütunun kaç farklı kategori içerdiğini anlamamıza yardımcı olur. Bu, özellikle modelleme aşamasında hangi sütunların one-hot encoding veya label encoding gerektirebileceğini belirlemek için önemlidir.
# Ayrıca, benzersiz değerlerin sayısı çok yüksek olan kategorik sütunlar, modelin karmaşıklığını artırabilir ve overfitting riskini yükseltebilir. Bu nedenle, benzersiz değerlerin sayısını bilmek, veri ön işleme stratejilerini planlamak için kritik bir adımdır.
# Sonuç olarak, benzersiz değerlerin sayısını göstermek, veri setindeki kategorik sütunların yapısını ve modelleme sürecinde nasıl ele alınması gerektiğini anlamamıza yardımcı olur.

df_uniques = pd.DataFrame(
    data=[[col, len(df[col].unique())]for col in df.columns], # Her bir sütun için benzersiz değerlerin sayısını hesaplar ve bir liste oluşturur. Bu liste, her sütun adı ve o sütundaki benzersiz değerlerin sayısını içeren alt listelerden oluşur.
    columns=['Variable Name', 'Unique Values Count'] # Oluşturulan listeyi bir DataFrame'e dönüştürür ve sütun adlarını 'Variable Name' ve 'Unique Values Count' olarak belirler.
).set_index('Variable Name') # 'Variable Name' sütununu DataFrame'in indeksine dönüştürür, böylece her bir satırın indeks olarak sütun adını kullanır ve 'Unique Values Count' sütunu benzersiz değerlerin sayısını içerir.

df_uniques # Her bir sütunun benzersiz değer sayısını gösterir.

,Unique Values Count
Variable Name,
months,72
offer,6
multiple,2
internet_type,4
gb_mon,50
security,2
backup,2
protection,2
support,2


In [38]:
# İkili kategorik değişkenler, modelleme sürecinde genellikle daha basit işlemler gerektirir, bu nedenle bu tür değişkenlerin sayısını bilmek, veri ön işleme stratejilerini planlamak için önemlidir.
# Ayrıca, ikili kategorik değişkenler, modelin performansını etkileyebilir, bu nedenle bu tür değişkenlerin sayısını bilmek, modelleme sürecinde nasıl ele alınması gerektiğini anlamamıza yardımcı olur.

binary_variables = list(df_uniques[df_uniques['Unique Values Count'] == 2].index) # Benzersiz değer sayısı 2 olan sütunları seçer ve bu sütunların adlarını bir liste olarak saklar. 
# Bu, ikili (binary) kategorik değişkenleri tanımlamak için kullanılır, çünkü bu tür değişkenler sadece iki farklı kategori içerir (örneğin, 'Evet'/'Hayır', '0'/'1' gibi). 
# Bu liste, daha sonra bu sütunlara özel işlemler uygulamak için kullanılabilir, örneğin one-hot encoding veya label encoding gibi.

binary_variables # İkili (binary) kategorik değişkenlerin listesini gösterir.

['multiple',
 'security',
 'backup',
 'protection',
 'support',
 'unlimited',
 'paperless',
 'churn_value']

In [39]:
# Çok kategorili değişkenler, modelin karmaşıklığını artırabilir, bu nedenle bu tür değişkenlerin sayısını bilmek, veri ön işleme stratejilerini planlamak için önemlidir.
# Ayrıca, çok kategorili değişkenler, modelin performansını etkileyebilir, bu nedenle bu tür değişkenlerin sayısını bilmek, modelleme sürecinde nasıl ele alınması gerektiğini anlamamıza yardımcı olur.

# Benzersiz değer sayısı 2'den büyük ve 6'dan küçük veya eşit olan sütunları seçer ve bu sütunların adlarını bir liste olarak saklar.
# Bu, çok kategorili (multiclass) kategorik değişkenleri tanımlamak için kullanılır, çünkü bu tür değişkenler birden fazla kategori içerir (örneğin, 'Kırmızı'/'Sarı'/'Yeşil' gibi).
# Bu liste, daha sonra bu sütunlara özel işlemler uygulamak için kullanılabilir, örneğin one-hot encoding veya label encoding gibi.
categorical_variables = list(df_uniques[(df_uniques['Unique Values Count'] > 2) & (df_uniques['Unique Values Count'] <= 6)].index)

categorical_variables # Çok kategorili (multiclass) kategorik değişkenlerin listesini gösterir.

['offer', 'internet_type', 'contract', 'payment', 'satisfaction']

In [40]:
# Çok kategorili değişkenlerin benzersiz değerlerini göstermek, her bir çok kategorili sütunun hangi kategorileri içerdiğini anlamamıza yardımcı olur. 
# Bu, özellikle modelleme aşamasında hangi sütunların one-hot encoding veya label encoding gerektirebileceğini belirlemek için önemlidir.

# Ayrıca, çok kategorili değişkenlerin benzersiz değerlerini bilmek, modelin karmaşıklığını artırabilecek kategorilerin sayısını ve türünü anlamamıza yardımcı olur. 
# Bu, veri ön işleme stratejilerini planlamak için kritik bir adımdır.

[[cat, list(df[cat].unique())]for cat in categorical_variables] # Çok kategorili değişkenlerin benzersiz değerlerini gösterir. Her bir çok kategorili sütun için, sütun adı ve o sütundaki benzersiz kategorilerin listesini içeren alt listelerden oluşan bir liste oluşturur.

[['offer', ['None', 'Offer E', 'Offer D', 'Offer C', 'Offer B', 'Offer A']],
 ['internet_type', ['DSL', 'Fiber Optic', 'Cable', 'None']],
 ['contract', ['Month-to-Month', 'One Year', 'Two Year']],
 ['payment', ['Bank Withdrawal', 'Credit Card', 'Mailed Check']],
 ['satisfaction', [3, 2, 1, 4, 5]]]

In [41]:
# KNN algoritması, sayısal verilerle çalışır, bu nedenle kategorik değişkenlerin sayısal verilere dönüştürülmesi gerekir.
# Kategorik değişkenleri sayısal verilere dönüştürmek için genellikle one-hot encoding veya label encoding gibi teknikler kullanılır.
# Ancak, bazı durumlarda, özellikle kategorik değişkenler ordinal (sıralı) ise, bu tür değişkenleri sayısal verilere dönüştürmek için pd.cut() fonksiyonu kullanılabilir. 
# Bu fonksiyon, sürekli bir değişkeni belirli aralıklara bölerek kategorik bir değişkene dönüştürür.
# Örneğin, 'months' adlı bir sütun varsa ve bu sütun sürekli bir değişken içeriyorsa, pd.cut() fonksiyonu kullanılarak bu sütun belirli aralıklara bölünebilir ve böylece kategorik bir değişkene dönüştürülebilir.
# Bu, KNN algoritması için uygun bir format sağlar ve modelin performansını artırabilir.

df['months'] = pd.cut(df['months'], bins=5) # 'months' sütununu 5 eşit aralığa bölerek kategorik bir değişkene dönüştürür. bins=5 ifadesi, 'months' sütunundaki değerleri 5 farklı kategoriye ayırır.

In [42]:
# 'months' sütunu, müşterilerin hizmette kalma süresini temsil eder ve bu süre, müşteri davranışını etkileyebilir. Bu nedenle, 'months' sütunu ordinal (sıralı) bir değişken olarak kabul edilir, çünkü daha uzun süre hizmette kalan müşteriler genellikle daha sadık ve değerli müşteriler olarak değerlendirilir.
# 'contract' sütunu, müşterilerin hizmet sözleşmesi türünü temsil eder ve bu tür sözleşmeler genellikle belirli bir süre boyunca hizmeti kullanmayı taahhüt eder. Bu nedenle, 'contract' sütunu da ordinal (sıralı) bir değişken olarak kabul edilir, çünkü daha uzun süreli sözleşmeler genellikle daha sadık müşterileri temsil eder.
# 'satisfaction' sütunu, müşterilerin hizmetten ne kadar memnun olduklarını temsil eder ve bu memnuniyet düzeyi genellikle belirli bir sıralamaya sahiptir (örneğin, 'Düşük', 'Orta', 'Yüksek'). Bu nedenle, 'satisfaction' sütunu da ordinal (sıralı) bir değişken olarak kabul edilir, çünkü daha yüksek memnuniyet düzeyleri genellikle daha sadık müşterileri temsil eder.  
# Bu nedenle, 'months', 'contract' ve 'satisfaction' sütunları ordinal (sıralı) değişkenler olarak kabul edilir ve bu sütunların sayısal verilere dönüştürülmesi için pd.cut() veya benzeri teknikler kullanılabilir.
# Bu sütunların sayısal verilere dönüştürülmesi, KNN algoritmasının bu bilgileri daha etkili bir şekilde kullanmasını sağlar ve modelin performansını artırabilir.

ordinal_variables = ['months', 'contract', 'satisfaction'] # 'months', 'contract' ve 'satisfaction' sütunlarını ordinal (sıralı) değişkenler olarak tanımlar. Bu, bu sütunların belirli bir sıralamaya sahip olduğunu ve bu sıralamanın modelleme sürecinde dikkate alınması gerektiğini belirtir.

In [43]:
# Kategorik ve ikili değişkenler tanımlandıktan sonra, geriye kalan sütunlar sayısal değişkenler olarak kabul edilir. Bu nedenle, sayısal değişkenlerin listesini oluşturmak için, tüm sütunlardan ordinal, kategorik ve ikili değişkenlerin çıkarılması gerekir.
# Bu, sayısal değişkenlerin doğru bir şekilde tanımlanmasını sağlar ve modelleme sürecinde bu değişkenlere uygun işlemler uygulanmasına olanak tanır.
# Ayrıca, sayısal değişkenlerin doğru bir şekilde tanımlanması, modelin performansını artırabilir, çünkü KNN algoritması sayısal verilerle daha etkili bir şekilde çalışır.

numeric_variables = list(
    set(df.columns) - set(ordinal_variables) - set(categorical_variables) - set(binary_variables) # Tüm sütunlardan ordinal, kategorik ve ikili değişkenlerin çıkarılmasıyla sayısal değişkenlerin listesini oluşturur.
) # set() fonksiyonu, her bir listeyi küme (set) veri yapısına dönüştürür ve bu kümeler arasındaki farkı alarak sayısal değişkenlerin listesini oluşturur. Bu, sayısal değişkenlerin doğru bir şekilde tanımlanmasını sağlar. 

numeric_variables # Sayısal değişkenlerin listesini gösterir. Bu liste, veri setindeki tüm sütunlardan ordinal, kategorik ve ikili değişkenlerin çıkarılmasıyla oluşturulur.

['gb_mon', 'monthly']

In [44]:
# KNN algoritması, sayısal verilerle çalışır, bu nedenle kategorik değişkenlerin sayısal verilere dönüştürülmesi gerekir.
# Kategorik değişkenleri sayısal verilere dönüştürmek için genellikle one-hot encoding veya label encoding gibi teknikler kullanılır.
# Ancak, bazı durumlarda, özellikle kategorik değişkenler ordinal (sıralı) ise, bu tür değişkenleri sayısal verilere dönüştürmek için pd.cut() fonksiyonu kullanılabilir.
# Bu fonksiyon, sürekli bir değişkeni belirli aralıklara bölerek kategorik bir değişkene dönüştürür. 
# Örneğin, 'months' adlı bir sütun varsa ve bu sütun sürekli bir değişken içeriyorsa, # pd.cut() fonksiyonu kullanılarak 
# bu sütun belirli aralıklara bölünebilir ve böylece kategorik bir değişkene dönüştürülebilir.

# one-hot encoding, her bir kategorik değeri ayrı bir sütun olarak temsil eder ve bu sütunlarda 0 veya 1 değerleri kullanarak kategorik değişkeni sayısal hale getirir. 
# Bu yöntem, özellikle ikili kategorik değişkenler için uygundur.

# label encoding, her bir kategorik değere benzersiz bir sayısal değer atar. Bu yöntem, özellikle ordinal (sıralı) kategorik değişkenler için uygundur, 
# çünkü bu tür değişkenlerde belirli bir sıralama vardır ve label encoding bu sıralamayı korur.

lb, oe = LabelBinarizer(), OrdinalEncoder() # LabelBinarizer ve OrdinalEncoder sınıflarını lb ve oe değişkenlerine atar. 
# LabelBinarizer, ikili kategorik değişkenleri one-hot encoding ile dönüştürmek için kullanılırken, 
# OrdinalEncoder ise ordinal (sıralı) kategorik değişkenleri sayısal verilere dönüştürmek için kullanılır.

for column in ordinal_variables: # Her bir ordinal değişken için döngü başlatır.
    df[column] = oe.fit_transform(df[[column]]) # OrdinalEncoder'ı kullanarak her bir ordinal değişkeni sayısal verilere dönüştürür. 
# fit_transform() yöntemi, önce veriye uyum sağlar (fit) ve ardından veriyi dönüştürür (transform). 
# Bu işlem, ordinal değişkenlerin KNN algoritması tarafından kullanılabilir hale gelmesini sağlar.

for column in binary_variables:# Her bir ikili kategorik değişken için döngü başlatır.
    df[column] = lb.fit_transform(df[column]) # LabelBinarizer'ı kullanarak her bir ikili kategorik değişkeni one-hot encoding ile dönüştürür. 
# fit_transform() yöntemi, önce veriye uyum sağlar (fit) ve ardından veriyi dönüştürür (transform). 
# Bu işlem, ikili kategorik değişkenlerin KNN algoritması tarafından kullanılabilir hale gelmesini sağlar.

In [45]:
# Çok kategorili değişkenlerin benzersiz değerlerini göstermek, her bir çok kategorili sütunun hangi kategorileri içerdiğini anlamamıza yardımcı olur. 
# Bu, özellikle modelleme aşamasında hangi sütunların one-hot encoding veya label encoding gerektirebileceğini belirlemek için önemlidir.
# Ayrıca, çok kategorili değişkenlerin benzersiz değerlerini bilmek, modelin karmaşıklığını artırabilecek kategorilerin sayısını ve türünü anlamamıza yardımcı olur. 
# Bu, veri ön işleme stratejilerini planlamak için kritik bir adımdır.
for col in categorical_variables: # Her bir çok kategorili değişken için döngü başlatır.
    # value_counts() yöntemi, her bir benzersiz değerin kaç kez göründüğünü sayar ve bu bilgiyi bir Series olarak döndürür. Bu, her bir kategorik değişkenin değerlerinin frekansını gösterir.
    freq = df[col].value_counts() # Her bir kategorik değişkenin değerlerini sayar. 
    print(freq) # Kategorik değişkenin değerlerini ve frekanslarını yazdırır.

None       3877
Offer B     824
Offer E     805
Offer D     602
Offer A     520
Offer C     415
Name: offer, dtype: int64
Fiber Optic    3035
DSL            1652
None           1526
Cable           830
Name: internet_type, dtype: int64
0.0    3610
2.0    1883
1.0    1550
Name: contract, dtype: int64
Bank Withdrawal    3909
Credit Card        2749
Mailed Check        385
Name: payment, dtype: int64
2.0    2665
3.0    1789
4.0    1149
0.0     922
1.0     518
Name: satisfaction, dtype: int64


In [46]:
# Çok kategorili değişkenlerin benzersiz değerlerini göstermek, her bir çok kategorili sütunun hangi kategorileri içerdiğini anlamamıza yardımcı olur.
# Bu, özellikle modelleme aşamasında hangi sütunların one-hot encoding veya label encoding gerektirebileceğini belirlemek için önemlidir.
# Ayrıca, çok kategorili değişkenlerin benzersiz değerlerini bilmek, modelin karmaşıklığını artırabilecek kategorilerin sayısını ve türünü anlamamıza yardımcı olur.
# Bu, veri ön işleme stratejilerini planlamak için kritik bir adımdır.

for col in categorical_variables: # Her bir çok kategorili değişken için döngü başlatır.
    freq = df[col].value_counts() # Her bir kategorik değişkenin değerlerini sayar. 
    df[col] = df[col].map(freq) # Her bir kategorik değişkenin değerlerini frekanslarıyla değiştirir.

df.head()

,months,offer,multiple,internet_type,gb_mon,security,backup,protection,support,unlimited,contract,paperless,payment,monthly,satisfaction,churn_value
0,0.0,3877,0,1652,8,0,0,1,0,0,3610,1,3909,39.65,2665,1
1,0.0,805,1,3035,17,0,1,0,0,1,3610,1,2749,80.65,2665,1
2,1.0,602,1,3035,52,0,0,0,0,1,3610,1,3909,95.45,518,1
3,1.0,415,0,3035,12,0,1,1,0,1,3610,1,3909,98.50,518,1
4,2.0,415,1,3035,14,0,0,0,0,1,3610,1,3909,76.50,518,1


In [47]:
def detect_outliers_iqr(df: pd.DataFrame, factor=1.5): # Sayısal değişkenlerdeki aykırı değerleri tespit etmek için Interquartile Range (IQR) yöntemini kullanır.
    outlier_flags = pd.DataFrame(index=df.index) # Aykırı değerleri tespit etmek için kullanılacak bir DataFrame oluşturur. 
    # Bu DataFrame, df ile aynı indekslere sahip olacak şekilde başlatılır, ancak başlangıçta boş olacaktır.
    
    for column in df.columns: # Her bir sütun için döngü başlatır.
        Q1 = df[column].quantile(0.25) # Her bir sütunun 1. çeyreklik değerini (Q1) hesaplar. Bu, verinin alt %25'lik kısmını temsil eder.
        Q3 = df[column].quantile(0.75) # Her bir sütunun 3. çeyreklik değerini (Q3) hesaplar. Bu, verinin üst %25'lik kısmını temsil eder.
        IQR = Q3 - Q1 # Interquartile Range (IQR) hesaplanır. Bu, verinin orta %50'lik kısmının yayılımını temsil eder.
    
        lower_bound = Q1 - factor * IQR # Alt sınır hesaplanır. Bu, aykırı değerlerin alt sınırını belirler.
        upper_bound = Q3 + factor * IQR # Üst sınır hesaplanır. Bu, aykırı değerlerin üst sınırını belirler.
        outlier_flag = (df[column] < lower_bound) | (df[column] > upper_bound) # Aykırı değerler tespit edilir.
        outlier_flags[column] = outlier_flag # Aykırı değerler DataFrame'ine eklenir.
    
    outlier_counts = outlier_flags.sum() # Her bir sütun için aykırı değerlerin sayısını hesaplar. 
    # Bu, her bir sayısal değişkende kaç tane aykırı değer olduğunu gösterir.

    outlier_rows = df[outlier_flags.any(axis=1)] # Aykırı değer içeren satırları seçer. 
    # Bu, hangi gözlemlerin aykırı değerlere sahip olduğunu anlamamıza yardımcı olur ve bu gözlemler üzerinde nasıl işlem yapacağımızı planlamamıza olanak tanır 
    # (örneğin, aykırı değerleri kaldırmak, dönüştürmek veya modellemeye dahil etmek gibi).

    return outlier_counts, outlier_rows # Aykırı değerlerin sayısını ve aykırı değer içeren satırları döndürür. 
    # Bu bilgiler, veri setindeki aykırı değerlerin etkisini değerlendirmek ve bu gözlemler üzerinde nasıl işlem yapacağımızı planlamak için kullanılır.

In [48]:
outlier_counts, outlier_rows = detect_outliers_iqr(df[numeric_variables]) # Sayısal değişkenlerdeki aykırı değerleri tespit eder.

print("Outlier Counts:\n", outlier_counts) # Her bir sayısal değişken için tespit edilen aykırı değerlerin sayısını gösterir.
print("Outlier Rows:\n", outlier_rows) # Aykırı değer içeren satırları gösterir. Bu, hangi gözlemlerin aykırı değerlere sahip olduğunu anlamamıza yardımcı olur ve bu gözlemler üzerinde nasıl işlem yapacağımızı planlamamıza olanak tanır (örneğin, aykırı değerleri kaldırmak, dönüştürmek veya modellemeye dahil etmek gibi).

Outlier Counts:
 gb_mon     362
monthly      0
dtype: int64
Outlier Rows:
       gb_mon  monthly
700       75   104.50
1147      69    55.20
1165      71    74.40
1175      73    94.85
1178      76    79.25
...      ...      ...
6975      73    56.75
6976      85   104.15
6997      76    43.05
6999      85    64.35
7006      76    83.20

[362 rows x 2 columns]


In [49]:
df.columns

Index(['months', 'offer', 'multiple', 'internet_type', 'gb_mon', 'security',
       'backup', 'protection', 'support', 'unlimited', 'contract', 'paperless',
       'payment', 'monthly', 'satisfaction', 'churn_value'],
      dtype='object')

In [50]:
binary_variables

['multiple',
 'security',
 'backup',
 'protection',
 'support',
 'unlimited',
 'paperless',
 'churn_value']

In [51]:
# pipeline, veri ön işleme adımlarını düzenli ve okunabilir bir şekilde uygulamak için kullanılan bir yapıdır.

robust_cols = ['gb_mon', 'payment'] # Aykırı değerlere duyarlı olan sayısal sütunları tanımlar. Bu sütunlar, aykırı değerlerin etkisini azaltmak için RobustScaler ile ölçeklendirilecektir.
min_max_cols = ['months', 'monthly', 'satisfaction'] # Aykırı değerlere duyarlı olmayan sayısal sütunları tanımlar. Bu sütunlar, MinMaxScaler ile ölçeklendirilecektir.
standard_cols = ['offer', 'internet_type', 'contract'] # Aykırı değerlere duyarlı olmayan sayısal sütunları tanımlar. Bu sütunlar, StandardScaler ile ölçeklendirilecektir.

# ColumnTransformer, farklı sütunlara farklı ölçeklendirme yöntemleri uygulamak için kullanılır.
preprocessor = ColumnTransformer(
    transformers=[
        ('robust', RobustScaler(), robust_cols), # 'robust' adı verilen bir transformer, RobustScaler'ı robust_cols listesinde belirtilen sütunlara uygular. 
        ('minmax', MinMaxScaler(), min_max_cols), # 'minmax' adı verilen bir transformer, MinMaxScaler'ı min_max_cols listesinde belirtilen sütunlara uygular.
        ('standard', StandardScaler(), standard_cols), # 'standard' adı verilen bir transformer, StandardScaler'ı standard_cols listesinde belirtilen sütunlara uygular.
    ], # transformers parametresi, her bir ölçeklendirme yöntemini ve bu yöntemin uygulanacağı sütunları tanımlar.
    remainder='passthrough' # transformers parametresinde belirtilmeyen sütunların orijinal hallerini koruyarak işlem görmeden geçmesini sağlar. Bu, sadece belirli sütunlara ölçeklendirme uygulamak istediğimiz durumlarda kullanışlıdır.
)

# RobustScaler, aykırı değerlere duyarlı olan sütunlar için kullanılır ve bu sütunlardaki aykırı değerlerin etkisini azaltır.
# MinMaxScaler, veriyi 0 ile 1 arasında ölçeklendirir ve bu sütunlarda aykırı değerlerin etkisi daha azdır.
# StandardScaler, veriyi ortalaması 0 ve standart sapması 1 olacak şekilde ölçeklendirir ve bu sütunlarda aykırı değerlerin etkisi daha azdır.

# Bu yapı, veri setindeki farklı türdeki sayısal sütunlara uygun ölçeklendirme yöntemlerini uygulamamıza olanak tanır ve modelin performansını artırabilir.

scaled_data = preprocessor.fit_transform(df) # Veriyi ölçeklendirir. 
# fit_transform() yöntemi, önce veriye uyum sağlar (fit) ve ardından veriyi dönüştürür (transform).

In [52]:
scaled_columns = robust_cols + min_max_cols + standard_cols # Ölçeklendirme uygulanan sütunların adlarını birleştirir. 
# Bu, ölçeklendirme işlemi sonucunda hangi sütunların dönüştürüldüğünü takip etmek için önemlidir.

new_columns = (
    scaled_columns +
    [col for col in df.columns if col not in scaled_columns]
) # Yeni sütun adlarını oluşturur. 
# scaled_columns listesinde yer alan sütun adlarını öncelikli olarak ekler ve ardından scaled_columns listesinde yer almayan diğer sütunları ekler.

scaled_df = pd.DataFrame(
    data=scaled_data,
    columns=new_columns
) # Ölçeklendirilmiş veriyi yeni bir DataFrame'e dönüştürür.

scaled_df.head()

,gb_mon,payment,months,monthly,satisfaction,offer,internet_type,contract,multiple,security,backup,protection,support,unlimited,paperless,churn_value
0,-0.375000,0.0,0.00,0.212935,1.0,0.901760,-0.570172,0.967848,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0
1,0.000000,-1.0,0.00,0.620896,1.0,-1.021875,1.101197,0.967848,1.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0
2,1.458333,0.0,0.25,0.768159,0.0,-1.148990,1.101197,0.967848,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
3,-0.208333,0.0,0.25,0.798507,0.0,-1.266087,1.101197,0.967848,0.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
4,-0.125000,0.0,0.50,0.579602,0.0,-1.266087,1.101197,0.967848,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0


In [53]:
round(scaled_df.describe().T, 3) # Ölçeklendirilmiş verinin temel istatistiklerini gösterir. round() fonksiyonu, istatistiklerin daha okunabilir olması için sonuçları 3 ondalık basamağa yuvarlar.

,count,mean,std,min,25%,50%,75%,max
gb_mon,7043.0,0.146,0.851,-0.708,-0.583,0.000,0.417,2.833
payment,7043.0,-0.556,0.765,-3.038,-1.000,0.000,0.000,0.000
months,7043.0,0.434,0.398,0.000,0.000,0.250,0.750,1.000
monthly,7043.0,0.463,0.299,0.000,0.172,0.518,0.712,1.000
satisfaction,7043.0,0.601,0.353,0.000,0.294,0.592,1.000,1.000
offer,7043.0,-0.000,1.000,-1.266,-1.022,0.902,0.902,0.902
internet_type,7043.0,0.000,1.000,-1.564,-0.722,-0.570,1.101,1.101
contract,7043.0,0.000,1.000,-1.211,-0.859,0.968,0.968,0.968
multiple,7043.0,0.422,0.494,0.000,0.000,0.000,1.000,1.000
security,7043.0,0.287,0.452,0.000,0.000,0.000,1.000,1.000


In [54]:
y = scaled_df['churn_value'] # Hedef değişkeni (churn_value) y olarak tanımlar. Bu, modelin tahmin etmeye çalışacağı değişkendir.
X = scaled_df.drop(columns='churn_value') # Özellikleri (hedef değişken dışında kalan sütunlar) X olarak tanımlar.

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
) # Veriyi eğitim ve test setlerine böler.
# test_size=0.2 ifadesi, verinin %20'sinin test seti olarak ayrılacağını belirtir. 
# random_state=42 ifadesi ise, veri bölme işleminin tekrarlanabilir olmasını sağlar, böylece aynı kodu çalıştırdığınızda aynı eğitim ve test setlerini elde edersiniz.

print(
    f'Train Sets: {X_train.shape} - {y_train.shape}\n'
    f'Test Sets: {X_test.shape} - {y_test.shape}'
) # Eğitim ve test setlerinin boyutlarını gösterir. 
# X_train.shape ve y_train.shape, eğitim setindeki özellikler ve hedef değişkenin boyutlarını gösterirken, 
# X_test.shape ve y_test.shape ise test setindeki özellikler ve hedef değişkenin boyutlarını gösterir. 
# Bu bilgiler, veri bölme işleminin doğru bir şekilde yapıldığını doğrulamak için önemlidir.

Train Sets: (5634, 15) - (5634,)
Test Sets: (1409, 15) - (1409,)


KNeighborsClassifier, K-en yakın komşu algoritmasını kullanarak sınıflandırma yapan bir modeldir.

**n_neighbors** parametresi, sınıflandırma kararını verirken kaç komşunun dikkate alınacağını belirler. 
- Bu değer, modelin esnekliğini ve genelleme yeteneğini etkiler. 
  - Genellikle, daha düşük bir `n_neighbors` değeri modelin daha esnek olmasına ve eğitim verisine daha iyi uyum sağlamasına neden olabilir, ancak bu durum aşırı öğrenmeye (overfitting) yol açabilir.
  - Daha yüksek bir `n_neighbors` değeri ise modelin daha genelleştirilebilir olmasına yardımcı olabilir, ancak bu durum modelin eğitim verisine yeterince uyum sağlamamasına (underfitting) neden olabilir. 
> Genellikle, `n_neighbors` değeri 3 ile 10 arasında seçilir (default değeri 5'tir), ancak en iyi performansı sağlamak için çapraz doğrulama (cross-validation) kullanılarak optimize edilebilir.

**weights** parametresi, komşuların sınıflandırma kararına katkısını (komşuların hedef değişken değerlerini nasıl ağırlıklandıracağını) belirler.
- 'uniform' seçeneği, tüm komşuların eşit ağırlıkta olduğunu varsayar, yani her komşu sınıflandırma kararına aynı derecede katkıda bulunur. 
  - modelin tahmin yaparken tüm komşuların hedef değişken değerlerini aynı şekilde dikkate alacağı anlamına gelir. 
- 'distance' seçeneği, komşuların sınıflandırma kararına katkısının, komşuların hedef örneğe olan uzaklığına bağlı olduğunu varsayar. Bu durumda, daha yakın komşular daha yüksek ağırlık alır ve sınıflandırma kararına daha fazla katkıda bulunur. 
  - Modelin tahmin yaparken daha yakın komşuların hedef değişken değerlerini daha fazla dikkate alacağı anlamına gelir. 
  - 'distance' seçeneği, özellikle veri setinde gürültü (noise) varsa veya bazı komşuların hedef örneğe çok yakın olduğu durumlarda daha iyi performans gösterebilir, çünkü bu durumda daha yakın komşuların sınıflandırma kararına daha fazla katkıda bulunması mantıklı olabilir.
  - 'distance' seçeneği, özellikle komşuların hedef değişken değerlerinin birbirinden farklı olduğu durumlarda, modelin tahmin performansını artırabilir.
> Genellikle, weights parametresi 'uniform' veya 'distance' olarak seçilir (default değeri 'uniform'dur), ancak en iyi performansı sağlamak için çapraz doğrulama (cross-validation) kullanılarak optimize edilebilir.

**leaf_size** parametresi, KNN algoritmasının kullandığı veri yapısında (örneğin, KD-tree veya Ball-tree) yaprak düğümünde bulunacak maksimum örnek sayısını belirler. Bu değer, komşu arama algoritmasının performansını etkiler.
- Daha küçük bir leaf_size değeri, daha fazla yaprak düğümü oluşturur ve bu da komşu arama süresini azaltabilir, ancak daha fazla bellek kullanımına neden olabilir. 
- Daha büyük bir leaf_size değeri ise daha az yaprak düğümü oluşturur ve bu da bellek kullanımını azaltabilir, ancak komşu arama süresini artırabilir. 
> Genellikle, leaf_size değeri 20 ile 40 arasında seçilir (default değeri 30'dur), ancak en iyi performansı sağlamak için çapraz doğrulama (cross-validation) kullanılarak optimize edilebilir.

**p** parametresi, Minkowski mesafe metriği için kullanılan güç parametresini belirler.
- p değeri, modelin komşular arasındaki mesafeyi nasıl hesaplayacağını belirler ve bu da modelin performansını etkileyebilir. 
  - Örneğin, p=1, özellikle yüksek boyutlu verilerde daha iyi performans gösterebilir, çünkü Manhattan mesafesi, yüksek boyutlu verilerde daha az etkilenir. 
- p=1, Manhattan mesafesini (L1 normu) kullanır.
- p=2 ise Öklid mesafesini (L2 normu) kullanır.
  - p=2 özellikle düşük boyutlu verilerde daha iyi performans gösterebilir, çünkü Öklid mesafesi, düşük boyutlu verilerde daha etkili olabilir. 
> Genellikle, p değeri 1 veya 2 olarak seçilir (default değeri 2'dir), ancak en iyi performansı sağlamak için çapraz doğrulama (cross-validation) kullanılarak optimize edilebilir.

**metric** parametresi, komşular arasındaki mesafeyi hesaplamak için kullanılan metriği belirler.
- 'minkowski' seçeneği, Minkowski mesafe metriğini kullanır ve p parametresi ile birlikte çalışır. 
  - Minkowski mesafesi, p parametresine bağlı olarak farklı mesafe türlerini hesaplayabilir (örneğin, p=1 için Manhattan mesafesi, p=2 için Öklid mesafesi).
- 'euclidean' seçeneği, Öklid mesafesini kullanır ve p parametresine bağlı değildir. Bu seçenek, özellikle düşük boyutlu verilerde daha iyi performans gösterebilir, çünkü Öklid mesafesi, düşük boyutlu verilerde daha etkili olabilir.
- 'manhattan' seçeneği, Manhattan mesafesini kullanır ve p parametresine bağlı değildir. Bu seçenek, özellikle yüksek boyutlu verilerde daha iyi performans gösterebilir, çünkü Manhattan mesafesi, yüksek boyutlu verilerde daha az etkilenir. 
> Genellikle, metric parametresi 'minkowski', 'euclidean' veya 'manhattan' olarak seçilir (default değeri 'minkowski'dir), ancak en iyi performansı sağlamak için çapraz doğrulama (cross-validation) kullanılarak optimize edilebilir.   

**n_jobs** parametresi, komşu arama işlemi için kullanılacak CPU çekirdeği sayısını belirler.
- n_jobs=-1 değeri, tüm çekirdeklerin kullanılmasını sağlar, bu da büyük veri setlerinde komşu arama süresini önemli ölçüde azaltabilir. Ancak, bu durum, diğer işlemlerin performansını etkileyebilir, çünkü tüm çekirdekler komşu arama işlemi için kullanılır. 
- n_jobs=1 olarak ayarlandığında, KNN algoritması tek bir işlemci çekirdeği kullanır. Bu, küçük veri setlerinde yeterli olabilir ve sistem kaynaklarını daha az kullanır.
> Genellikle, n_jobs paremetresi -1 olarak seçilir (default değeri -1'dir), ancak en iyi performansı sağlamak için veri setinin büyüklüğüne, sistem kaynaklarına ve diğer işlemlerin performansına bağlı olarak optimize edilebilir.

**algorithm** parametresi, KNN algoritmasının hangi yöntemi kullanarak en yakın komşuları bulacağını belirtir.
- ball_tree, kd_tree ve brute algoritmaları arasından seçim yapabilir. 
  - Ball_tree ve kd_tree, büyük veri setlerinde daha hızlı komşu arama sağlar, ancak küçük veri setlerinde brute algoritması daha iyi performans gösterebilir.
  - ball_tree algoritması, yüksek boyutlu verilerde daha iyi performans gösterebilirken, kd_tree algoritması düşük boyutlu verilerde daha hızlı olabilir. 
  - Brute algoritması ise tüm veri setini tarayarak komşuları bulur ve küçük veri setlerinde etkili olabilir.
- 'auto' seçeneği, uygun algoritmayı otomatik olarak seçer. Bu değer, veri setinin boyutuna ve özelliklerine bağlı olarak en iyi performansı sağlayacak algoritmayı seçer.
> Genellikle, algorithm parametresi 'auto' olarak seçilir (default değeri 'auto'dur), ancak en iyi performansı sağlamak için veri setinin yapısına bağlı olarak optimize edilebilir.

In [55]:
# KNN sınıflandırıcısını oluşturur ve eğitim verisi üzerinde eğitir.
# KNN algoritması, sınıflandırma problemlerinde kullanılan bir algoritmadır ve bu algoritmanın performansı, verinin doğru bir şekilde ölçeklendirilmesine bağlıdır.

# n_neighbors parametresi, sınıflandırma kararını verirken kaç komşunun dikkate alınacağını belirler.
# weights parametresi, komşuların sınıflandırma kararına katkısını belirler.
# leaf_size parametresi, yapının yaprak düğümlerinde kaç örneğin bulunacağını belirler. Bu değer, komşu arama algoritmasının performansını etkiler.
# p parametresi, Minkowski mesafe metriği için kullanılan güç parametresini belirler.
# metric parametresi, komşular arasındaki mesafeyi hesaplamak için kullanılan metriği belirler.
# n_jobs parametresi, komşu arama işlemi için kullanılacak CPU çekirdeklerinin sayısını belirler. 
# Bu, komşu arama işleminin paralel olarak gerçekleştirilmesine olanak tanır ve bu da büyük veri setlerinde işlem süresini azaltabilir.

# KNeighborsClassifier, K-en yakın komşu algoritmasını kullanarak sınıflandırma yapan bir modeldir.
knn_v1 = KNeighborsClassifier() # KNN sınıflandırıcısını oluşturur. 
# KNeighborsClassifier, K-Nearest Neighbors algoritmasını uygulamak için kullanılan bir sınıflandırıcıdır.
# Bu modelde n_neighbors değeri 5 olarak belirlenmiştir, bu da sınıflandırma kararını verirken en yakın 5 komşunun dikkate alınacağı anlamına gelir.

knn_v1 = knn_v1.fit(X_train, y_train) # Eğitim verisi üzerinde KNN sınıflandırıcısını eğitir.
# Bu model, eğitim verisi üzerinde fit edilir, böylece model öğrenir ve daha sonra test verisi üzerinde tahminler yapabilir.

y_pred = knn_v1.predict(X_test) # Test verisi üzerinde tahminler yapar.

classification_report, modelin performansını değerlendirmek için kullanılan bir araçtır.
- modelin her bir sınıf için **precision, recall, f1-score** ve **support** değerlerini gösterir.
> Bu rapor, modelin her bir sınıf için performansını detaylı bir şekilde değerlendirmeye yardımcı olur.
---
**0.0** doğruluk, modelin hiçbir doğru tahmin yapmadığını gösterirken, **1.0** doğruluk, modelin tüm tahminlerinin doğru olduğunu gösterir.
> Precision, recall ve F1-score değerleri de 0.0 ile 1.0 arasında değişir, bu değerler ne kadar yüksekse modelin o sınıfı o kadar iyi tahmin ettiği anlamına gelir. Bu metrikler, modelin performansını değerlendirmek ve modelin hangi sınıfları daha iyi tahmin ettiğini anlamak için kritik öneme sahiptir.
---
**Precision**, modelin doğru pozitif tahminlerinin toplam pozitif tahminlere oranını gösterir.
- Modelin pozitif olarak tahmin ettiği örneklerin ne kadarının gerçekten pozitif olduğunu gösterir.
> Daha yüksek bir precision değeri, modelin pozitif tahminlerinin daha doğru olduğunu gösterir, bu da özellikle yanlış pozitiflerin maliyetli olduğu durumlarda önemlidir.
---
**Recall**, modelin doğru pozitif tahminlerinin toplam gerçek pozitiflere oranını gösterir.
-  modelin gerçek pozitif örneklerin ne kadarını doğru bir şekilde tahmin ettiğini gösterir, bu da özellikle yanlış negatiflerin maliyetli olduğu durumlarda önemlidir, çünkü bu durumlarda modelin gerçek pozitif örneklerin daha fazlasını tanımlama yeteneği kritik olabilir.
> Daha yüksek bir recall değeri, modelin gerçek pozitif örneklerin daha fazlasını tanımlama yeteneğinin daha iyi olduğunu gösterir, bu da özellikle yanlış negatiflerin maliyetli olduğu durumlarda önemlidir. 
---
**f1-score**, precision ve recall'un harmonik ortalamasıdır ve modelin dengeli performansını değerlendirir.
- Ozellikle sınıf dengesizliği durumlarında önemli bir metriktir, çünkü bu durumlarda accuracy yanıltıcı olabilir.
> Daha yüksek bir F1-score değeri, modelin hem precision hem de recall açısından daha iyi performans gösterdiğini gösterir. 
---
**Support**, modelin her bir sınıf için ne kadar verisi olduğunu (her sınıfın gerçek örnek sayısını) gösterir ve bu bilgi, modelin performansını değerlendirmek için önemlidir.
- Özellikle, sınıf dengesizliği durumlarında, support değerleri modelin hangi sınıflar için daha fazla veri olduğunu ve bu durumun modelin performansını nasıl etkileyebileceğini anlamaya yardımcı olabilir. 
> Daha yüksek bir support değeri, modelin o sınıf için daha fazla veri olduğunu gösterir, bu da modelin o sınıf için daha iyi performans gösterebileceği anlamına gelebilir.
---
**Accuracy**, modelin doğru tahmin ettiği örneklerin oranını hesaplar. Ancak, sınıf dengesizliği durumlarında, model çoğunluk sınıfını tahmin ederek yüksek bir accuracy elde edebilir, bu da modelin performansını yanlış değerlendirmemize neden olabilir. Bu nedenle, sınıf dengesizliği durumlarında, precision, recall ve f1-score gibi diğer metrikleri de değerlendirmek önemlidir.
> Accuracy, modelin genel performansını değerlendirmek için yaygın olarak kullanılan bir metriktir, ancak sınıf dengesizliği durumlarında yanıltıcı olabilir.
> Accuracy, ne kadar yüksek olursa olsun, modelin performansını tam olarak değerlendirmek için precision, recall ve f1-score gibi diğer metrikleri de göz önünde bulundurmak önemlidir.

**macro avg**, modelin her bir sınıf için precision, recall ve f1-score değerlerinin basit ortalamasını gösterir. Bu, her bir sınıfın performansını eşit şekilde değerlendirmek için kullanılır ve sınıf dengesizliği durumlarında modelin genel performansını değerlendirmek için önemlidir.

**weighted avg**, modelin her bir sınıf için precision, recall ve f1-score değerlerinin, her sınıfın support değerleriyle ağırlıklandırılmış ortalamasını gösterir. Bu, sınıf dengesizliği durumlarında modelin genel performansını değerlendirmek için kullanılır ve sınıf dengesizliği durumlarında modelin performansını daha doğru bir şekilde değerlendirmek için önemlidir. 

In [56]:
# Model performansını değerlendirmek için çeşitli metrikler kullanılır.
# Classification Report, modelin doğruluk, precision, recall ve F1-score gibi metriklerini gösterir.
# Bu metrikler, modelin test verisi üzerindeki performansını değerlendirmek için kullanılır ve modelin hangi sınıfları daha iyi tahmin ettiğini anlamamıza yardımcı olur.
# Doğruluk (accuracy), modelin doğru tahminlerinin oranını gösterir. 
# Precision, modelin pozitif sınıflar için yaptığı tahminlerin ne kadarının doğru olduğunu gösterir. 
# Recall, modelin gerçek pozitif sınıfları ne kadar iyi yakaladığını gösterir. 
# F1-score, precision ve recall'un harmonik ortalamasıdır ve modelin genel performansını değerlendirmek için kullanılır.
# Bu metrikler, modelin hangi sınıfları daha iyi tahmin ettiğini anlamamıza yardımcı olur ve modelin performansını artırmak için hangi alanlarda iyileştirme yapılması gerektiğini belirlememize olanak tanır.
print(
    f'Classification Report:\n'
    f'{classification_report(y_test, y_pred)}\n'
)

Classification Report:
              precision    recall  f1-score   support

         0.0       0.87      0.90      0.89      1009
         1.0       0.73      0.67      0.70       400

    accuracy                           0.84      1409
   macro avg       0.80      0.79      0.79      1409
weighted avg       0.83      0.84      0.83      1409




Gerçek Durum;
- 100 hasta
- 900 sağlıklı

confusion_matrix sonucu:
1. TN (True Negative)  -> 850 sağlıklı kişiyi doğru tahmin etti
2. FN (False Negative) -> 20 hastayı yanlışlıkla sağlıklı tahmin etti
3. TP (True Positive)  -> 80 hastayı doğru tahmin etti
4. FP (False Positive) -> 50 sağlıklı kişiyi yanlışlıkla hasta tahmin etti